# Why inference costs what it does — batching, the KV-cache & quantization

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 39 §39.3 · type: concept-lab*

**The promise:** by the end you can explain *every line of a provider's pricing page* and reason about serving cost and latency from first principles — why long contexts cost more, why "batch" endpoints are cheaper, why a smaller or quantized model is faster — **without owning a GPU**.

Runs fully **offline and free**: pure in-notebook arithmetic and a tiny seeded simulator. No model is loaded, no network is touched, no key is needed. Picks up the two tiers you routed between in [`39-01-llm-gateway-routing-and-fallbacks.ipynb`](39-01-llm-gateway-routing-and-fallbacks.ipynb) and asks *why* one is cheaper than the other.

## 🧠 Why this matters

You don't need to run a GPU to *use* the internals of serving. The same four ideas that decide a self-hosting bill also explain things you see as a pure API consumer: why a 100K-token context costs and runs more than a one-line prompt, why providers sell a cheaper *batch* tier, and why a smaller or quantized model is both cheaper and faster.

Four levers explain most of inference performance — **batching**, the **KV-cache**, **quantization**, and the **throughput-vs-latency** trade-off. Understand them and a pricing page stops being magic: every number on it is one of these levers showing through. That understanding directly shapes how you design prompts, choose models, and architect for cost (Ch 40) — see §39.3.

## Objectives & prereqs

**By the end you can:**
- Predict the throughput multiple **continuous batching** buys over one-at-a-time serving.
- Compute **KV-cache** size from sequence length and explain why *it*, not the weights, caps concurrency.
- Apply the **VRAM rule of thumb** (≈2 bytes/param at 16-bit) and see what **quantization** to 8-/4-bit saves.
- Read the **throughput-vs-latency** knob and pick it per use case (chat vs batch job).
- Re-run the book's **TCO crossover** and see that *utilization*, not sticker price, governs break-even.

**Prereqs:** [`39-01`](39-01-llm-gateway-routing-and-fallbacks.ipynb) (the two tiers we explain here); Ch 8 (how LLMs work) read.

**Packages:** the standard library only (`random`, `math`, `dataclasses`). Nothing beyond `requirements.txt`.

**Runs free & offline by design.** Everything here is arithmetic and a toy simulator — there is no `MOCK=0` model path. (An *optional, heavy, non-CI* local-server appendix at the very end shows the real serving path; it is clearly flagged and skipped by default.)

In [ ]:
# --- Setup -------------------------------------------------------------------
import math
import os
import random
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; this notebook needs no keys at all

# This concept-lab is OFFLINE BY DESIGN: pure arithmetic + a seeded simulator, no
# model is ever loaded and no provider is called. MOCK stays 1 for consistency with
# the rest of the course and gates only the optional local-server appendix at the end.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(39)  # determinism for the batching simulator's arrival jitter

print(f"MOCK mode: {MOCK}  (this notebook is fully offline regardless)")
print("no GPU, no network, no key required.")

## 1 · 🧠 The four levers

Almost everything about serving performance is one of these four:

```
  BATCHING             many requests on the GPU together  >>  one at a time
  KV-CACHE             cache past keys/values; grows with sequence length
  QUANTIZATION         fewer bits per weight -> less memory, faster, tiny quality cost
  THROUGHPUT<->LATENCY  bigger batches lift total tok/s but raise per-request latency
```

We take them one at a time, each as a tiny computation you can poke. None of this requires a GPU — the *numbers* are what teach. (All dollar and token figures below are **illustrative assumptions**, not quoted prices.)

## 2 · Batching: the single biggest lever

GPUs are massively parallel, so serving requests *together* keeps the silicon busy; one-at-a-time leaves most of it idle between requests. **Continuous batching** (vLLM's signature trick) packs and unpacks requests dynamically as they arrive and finish, instead of waiting to assemble a fixed batch.

The toy below models a GPU that can process up to `BATCH_WIDTH` requests in parallel per *step*, each request needing some number of decode steps. We serve a set of requests two ways — strictly one-at-a-time, and continuously batched — and compare total wall-clock steps.

In [ ]:
@dataclass
class Req:
    rid: int
    steps: int  # decode steps this request needs (~ output tokens)


def make_requests(n=24, lo=20, hi=120):
    """A seeded set of requests with varied output lengths."""
    return [Req(i, random.randint(lo, hi)) for i in range(n)]


def serve_one_at_a_time(reqs):
    """No batching: the GPU finishes one request before starting the next."""
    return sum(r.steps for r in reqs)  # steps run back-to-back


def serve_continuous(reqs, batch_width=8):
    """Continuous batching: up to batch_width requests advance one step together.
    As a request finishes, a waiting one immediately takes its slot."""
    pending = sorted((r.steps for r in reqs))  # remaining steps per request
    steps_elapsed = 0
    active = []
    i = 0
    while i < len(pending) or active:
        # fill free slots from the queue
        while len(active) < batch_width and i < len(pending):
            active.append(pending[i]); i += 1
        # advance every active request by one decode step (the parallel part)
        steps_elapsed += 1
        active = [s - 1 for s in active]
        active = [s for s in active if s > 0]  # drop the finished ones
    return steps_elapsed


BATCH_WIDTH = 8
reqs = make_requests(n=24)
serial = serve_one_at_a_time(reqs)
batched = serve_continuous(reqs, batch_width=BATCH_WIDTH)
print(f"requests: {len(reqs)}   GPU batch width: {BATCH_WIDTH}")
print("total decode steps to drain the queue:")
print(f"  one-at-a-time : {serial:>5} steps")
print(f"  continuous    : {batched:>5} steps")
print(f"  speedup       : {serial / batched:>5.2f}x throughput")

## 🔮 Predict

Before reading the result above as final, predict: if you **double** the GPU's `BATCH_WIDTH` from 8 to 16, does the throughput multiple roughly **double** too? Or does it level off?

*Hint:* the queue has a fixed amount of work; batching can only overlap requests that actually exist at the same time. Write down "doubles" or "levels off," then run the next cell.

In [ ]:
for bw in (1, 2, 4, 8, 16, 32):
    b = serve_continuous(reqs, batch_width=bw)
    print(f"batch_width={bw:>2}  ->  {b:>4} steps   ({serial / b:5.2f}x vs serial)")
print()
print("Throughput rises fast at first, then FLATTENS: once the batch is wide")
print("enough to hold all concurrently-runnable work, wider does nothing here.")

**What you just saw.** Batching is the biggest lever in self-hosted serving — but it has diminishing returns. Throughput climbs steeply as the batch widens, then flattens once the batch is wide enough to hold all the work in flight; past that, extra width sits idle. This is *exactly* why providers can sell a cheaper **batch endpoint**: when they control scheduling and don't owe you low latency, they pack the GPU near its efficient width and pass the savings on. Same tokens, better packing, lower price.

## 3 · The KV-cache: why long contexts cost memory

As the model generates, it caches the attention **keys and values** for every token already processed, so it doesn't recompute them each step — that's what makes generation fast. The catch: this cache *grows linearly with sequence length*, and it, not the weights, is often what caps how many requests you can serve at once.

The size is pure arithmetic:

```
kv_bytes = 2 (K and V) x layers x seq_len x hidden_dim x bytes_per_value
```

Let's size it for a mid-range model and watch it climb with context length.

In [ ]:
@dataclass
class ModelShape:
    name: str
    layers: int
    hidden_dim: int
    params_billion: float


# A representative ~13B-class model shape (illustrative round numbers).
MODEL = ModelShape(name="mid-13B", layers=40, hidden_dim=5120, params_billion=13.0)
KV_BYTES_PER_VALUE = 2  # 16-bit K/V cache


def kv_cache_gb(model: ModelShape, seq_len: int, batch: int = 1) -> float:
    """KV-cache footprint in GB for `batch` sequences of length `seq_len`."""
    bytes_ = 2 * model.layers * seq_len * model.hidden_dim * KV_BYTES_PER_VALUE * batch
    return bytes_ / 1e9


print(f"model: {MODEL.name}  ({MODEL.layers} layers, hidden={MODEL.hidden_dim})")
print(f"{'context tokens':>16} {'KV-cache (1 req)':>20}")
for seq in (1_000, 4_000, 16_000, 64_000, 128_000):
    print(f"{seq:>16,} {kv_cache_gb(MODEL, seq):>17.2f} GB")

**This is why long contexts cost memory.** The KV-cache for one 128K-token request is *gigabytes*. Now flip it around: given a fixed VRAM budget left over after the weights, the KV-cache is what decides **how many requests fit at once** — your concurrency ceiling. The next cell shows that ceiling collapsing as context grows.

In [ ]:
KV_BUDGET_GB = 20.0  # VRAM left for KV after weights + activations (illustrative)

print(f"KV budget for concurrent requests: {KV_BUDGET_GB:.0f} GB")
print(f"{'context tokens':>16} {'per-req KV':>12} {'max concurrent':>16}")
for seq in (1_000, 4_000, 16_000, 64_000, 128_000):
    per = kv_cache_gb(MODEL, seq)
    concurrent = int(KV_BUDGET_GB / per)
    print(f"{seq:>16,} {per:>9.2f} GB {concurrent:>16,}")
print()
print("Long contexts don't just cost more tokens -- they evict concurrency:")
print("the KV-cache, not the weights, is the binding constraint at scale.")

## 4 · The VRAM rule of thumb & quantization

The weights themselves follow a famous back-of-the-envelope rule: **~2 bytes per parameter at 16-bit precision**. So a 13B model needs roughly `13e9 x 2 = 26 GB` just for weights, before any KV-cache or activations. Big models span multiple GPUs.

**Quantization** stores each weight at lower precision — 8-bit or 4-bit instead of 16 — shrinking the footprint (and speeding inference) for a *usually-small* quality cost. It's what lets a big model fit on a smaller GPU. The arithmetic is direct.

In [ ]:
def weights_gb(params_billion: float, bits: int) -> float:
    """VRAM for weights at a given precision. 16-bit -> ~2 bytes/param."""
    bytes_per_param = bits / 8
    return params_billion * 1e9 * bytes_per_param / 1e9


print(f"model: {MODEL.name}  ({MODEL.params_billion:.0f}B params)")
print(f"{'precision':>10} {'bytes/param':>12} {'weights VRAM':>14}  {'fits on...':>22}")
GPUS = [("24 GB (1x consumer)", 24), ("48 GB (1x pro)", 48), ("80 GB (1x datacenter)", 80)]
for bits, label in [(16, "16-bit"), (8, "8-bit"), (4, "4-bit")]:
    gb = weights_gb(MODEL.params_billion, bits)
    fits = next((name for name, cap in GPUS if gb <= cap), "needs multiple GPUs")
    print(f"{label:>10} {bits / 8:>11.2f}  {gb:>11.1f} GB  {fits:>22}")
print()
print("16-bit needs ~26 GB (spills a 24 GB card); 4-bit shrinks it to ~6.5 GB --")
print("the same model now fits a consumer GPU. Caveat: a small, measurable")
print("quality drop -- quantization trades a little accuracy for a lot of memory.")

## 5 · Throughput vs latency: the knob you actually turn

Bigger batches raise **total throughput** (tokens/sec across all users) but can raise **individual** latency — a request waits its turn in a wider batch and shares the GPU. They trade off, and the right setting depends on the workload:

- **User-facing chat** → optimize *per-request latency* (small batches, responsive).
- **Batch document job** → optimize *throughput* (wide batches, who cares about one doc's latency).

The toy below makes the trade-off visible: as batch size grows, aggregate tokens/sec rises while the time an individual request waits also rises.

In [ ]:
STEP_SECONDS = 0.010  # wall-clock per decode step for the whole batch (illustrative)
TOKENS_PER_STEP = 1     # one token per request per step


def throughput_and_latency(batch_size: int, tokens_per_req: int = 64):
    """Aggregate tokens/sec vs per-request latency at a given batch size."""
    # One step advances every request in the batch by one token, in STEP_SECONDS.
    agg_tokens_per_sec = batch_size * TOKENS_PER_STEP / STEP_SECONDS
    # A request still needs tokens_per_req steps; latency is steps x step time,
    # plus mild queueing that grows with batch size (sharing the GPU).
    queue_penalty = 1.0 + 0.04 * batch_size
    per_req_latency = tokens_per_req * STEP_SECONDS * queue_penalty
    return agg_tokens_per_sec, per_req_latency


print(f"{'batch':>6} {'aggregate tok/s':>18} {'per-req latency':>18}")
for bs in (1, 2, 4, 8, 16, 32, 64):
    tps, lat = throughput_and_latency(bs)
    print(f"{bs:>6} {tps:>15,.0f}/s {lat * 1000:>15.0f} ms")
print()
print("Throughput climbs ~linearly with batch; per-request latency climbs too.")
print("Chat picks the top of the table; a nightly batch job picks the bottom.")

## 6 · Two concepts to *know*, not run

Some serving ideas are worth understanding even though there's nothing meaningful to simulate offline.

**Speculative decoding.** Generation is sequential — one token at a time — and that, not raw compute, is what makes it slow. A small fast **draft model** proposes several tokens ahead; the large **target model** verifies that whole block in a single forward pass, keeping every token it would have produced anyway and discarding the rest. Because verifying *k* tokens costs about as much as generating one, the accepted tokens are nearly free throughput — at *no quality cost*, since the output is identical to what the target would have emitted alone. The payoff depends entirely on the **acceptance rate**: high on boilerplate and structured formats, low on surprising text (where you pay for the draft model with little to show).

**GPU cold starts & the warm floor.** Idle GPUs are pure waste, so you'd love to **scale to zero** between bursts — but starting a replica means pulling a big container and loading tens of GB of weights into VRAM, which takes *tens of seconds to minutes*. CPU services cold-start in milliseconds; a GPU replica does not. So the first requests after a scale-up sit in a queue while the replica warms, and tail latency spikes exactly when traffic does. The fix for latency-sensitive traffic is a **warm floor** of replicas (never truly zero) and autoscaling *above* it on a leading signal — queue depth or tokens-in-flight, not CPU — reserving scale-to-zero for traffic that tolerates a cold start (internal batch, dev endpoints).

## ⚠️ Pitfall: speculative *decoding* ≠ speculative *execution*

These share a word and live one layer apart — confusing them is a classic interview-and-design mistake.

| | **Speculative *decoding*** (this chapter) | **Speculative *execution*** (Ch 40) |
|---|---|---|
| Layer | *Inside* the decoder, token-level | *Application*-level |
| What | Draft proposes tokens, target verifies a block | Race two model tiers, or prefetch a likely next request |
| Output | **Mathematically identical** to the target alone | A *branch you might throw away* |
| Quality risk | **None** — guaranteed same distribution | The loser's work is wasted; you chose a result |

Speculative decoding is a guaranteed-correct trick *inside* the model; speculative execution is an application bet you can lose. Same word, different layer — don't let a design review blur them.

## 7 · ⚠️ Pitfall: "self-host is cheaper" — re-run the TCO crossover

The most expensive assumption in serving is that owning a GPU beats per-token pricing. It depends entirely on **utilization**, not the sticker prices. We re-run the book's worked crossover — **every number an illustrative assumption**, not a quote — and find the break-even volume *and* the capacity ceiling.

In [ ]:
# --- ALL NUMBERS BELOW ARE ILLUSTRATIVE ASSUMPTIONS, not quoted prices. ---
GPU_HOURLY = 3.00          # $/hr to rent one GPU node
BUSY_TOK_PER_SEC = 2_000   # sustained output tokens/sec when the GPU is busy
HOSTED_PER_MTOK = 3.00     # $ per 1M output tokens on a hosted API
SECONDS_PER_MONTH = 24 * 30 * 3600


def tco_crossover(utilization: float):
    monthly_node_cost = GPU_HOURLY * 24 * 30                       # paid 24/7 regardless
    effective_tok_per_sec = BUSY_TOK_PER_SEC * utilization         # real average capacity
    capacity_tok_month = effective_tok_per_sec * SECONDS_PER_MONTH
    breakeven_tok_month = monthly_node_cost / HOSTED_PER_MTOK * 1e6  # where hosted spend = node
    return monthly_node_cost, capacity_tok_month, breakeven_tok_month


print(f"node cost: ${GPU_HOURLY:.2f}/hr -> ${GPU_HOURLY * 24 * 30:,.0f}/month (paid whether busy or idle)")
print(f"{'utilization':>12} {'avg capacity/mo':>20} {'break-even/mo':>16}")
for util in (0.15, 0.30, 0.60, 0.90):
    cost, cap, be = tco_crossover(util)
    print(f"{util:>11.0%} {cap / 1e6:>16,.0f}M tok {be / 1e6:>13,.0f}M tok")
print()
print("Break-even (~720M tok/mo here) is FIXED by node cost / hosted price -- it does")
print("not move with utilization. What utilization changes is the CAPACITY ceiling:")
print("at 30% the node tops out ~1,550M tok/mo; at 60% it serves twice the traffic for")
print("the SAME $2,160, halving per-token cost. A lightly-loaded GPU is the most")
print("expensive option of all -- you pay full price for idle silicon.")

## 🎯 Senior lens

**Start hosted.** The right default for almost everyone is a hosted API: frontier quality, zero infrastructure, elastic scale, pay-per-token. Self-hosting is justified by specific forces — data residency, sustained high volume, a custom/fine-tuned model, or air-gapped operation — and notice that *"it seems cheaper"* is not on that list. At modest or bursty scale it rarely is, once you count GPU hours billed whether busy or idle, plus the engineering time to operate a serving stack.

So the senior move is to **instrument first**: measure your real token volume *and* your realistic utilization, then revisit self-hosting with actual numbers against the crossover above. Never architect a GPU fleet on day one for traffic you don't yet have. And when you *do* self-host, you drop it in behind the same gateway from [`39-01`](39-01-llm-gateway-routing-and-fallbacks.ipynb) — the four levers here tell you what that GPU will cost and how fast it will be (§39.1–39.3).

## Recap

- **Batching** is the biggest serving lever — continuous batching keeps the GPU busy — with diminishing returns once the batch holds all in-flight work. (It's why *batch endpoints* are cheaper.)
- The **KV-cache** grows linearly with sequence length; *it*, not the weights, caps concurrency — this is why **long contexts cost memory** and run slower.
- The **VRAM rule** is ~2 bytes/param at 16-bit (a 13B model ≈ 26 GB weights); **quantization** to 8-/4-bit shrinks that for a small quality cost, fitting big models on smaller GPUs.
- **Throughput and latency trade off**: wide batches lift aggregate tokens/sec but raise per-request latency — chat optimizes latency, batch jobs optimize throughput.
- **Speculative decoding** (token-level, guaranteed-correct) is *not* Ch 40's speculative **execution** (application-level, a branch you might discard).
- The **TCO crossover** is governed by *utilization*, not sticker price; a lightly-loaded self-hosted endpoint is often the most expensive option of all.

## Exercises

Each exercise *changes* a number and asks you to predict before you run. (Solutions arrive in `solutions/` in Phase 2.)

1. **Halve the KV bytes, double the concurrency?** In the KV-cache section, predict how `max_concurrent` at 64K tokens changes if you *halve* `KV_BYTES_PER_VALUE` to 1 (an 8-bit KV cache). Run and confirm, then explain why the KV-cache itself can be quantized.
2. **Find the batch sweet spot.** Vary `BATCH_WIDTH` over the request set in section 2 and find the smallest width that reaches within 5% of the maximum speedup. Predict it from the request-length distribution first.
3. **A 70B model on one card.** Use `weights_gb()` for a 70B model. Predict which precision (16/8/4-bit) first fits an 80 GB datacenter GPU *with* a 20 GB KV budget left over, then compute it.
4. **Move the break-even.** In the TCO cell, predict what happens to the break-even token volume if the hosted price *drops* to $1.50/Mtok while the GPU rent stays $3.00/hr. Run it; then state in one sentence why utilization still decides whether self-hosting wins.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

In [ ]:
# Exercise 4 -- your code here

## ⚠️ Optional appendix — the *real* serving path (heavy, non-CI, skipped by default)

Everything above was offline arithmetic. To *see* real serving, you can run a local model server and hit it **through the same gateway interface from [`39-01`](39-01-llm-gateway-routing-and-fallbacks.ipynb)** — proving "swap in your own server without touching application code." This is **opt-in and heavy**: it needs a local model pull (GB of weights) and ideally a GPU, runs nothing in CI, and incurs no cloud spend. Leave it commented unless you've installed a server.

```bash
# 1) Easiest: Ollama (CPU works; GPU is faster). Install from ollama.com, then:
#    ollama pull llama3.1:8b
#    ollama serve            # exposes an OpenAI-compatible endpoint on :11434

# 2) Or vLLM / TGI for the production path (needs a GPU):
#    vllm serve meta-llama/Llama-3.1-8B-Instruct   # OpenAI-compatible on :8000
```

```python
# OPT-IN ONLY -- uncomment when a local server is running. Reuses the gateway shape
# from 39-01: a backend whose .complete() points at the local OpenAI-compatible URL.
# Nothing in Gateway/route/cache changes -- that is the whole point.
#
# from openai import OpenAI
# local = OpenAI(base_url="http://localhost:11434/v1", api_key="not-needed")  # Ollama
# resp = local.chat.completions.create(
#     model="llama3.1:8b",
#     messages=[{"role": "user", "content": "Say hello in five words."}],
# )
# print(resp.choices[0].message.content)
```

The lesson lands even if you never run it: a self-hosted model is just another backend behind the `complete()` you already built — the gateway makes the serving boundary a config detail.

## Next

You can now reason about serving cost and latency from first principles. Where this goes:

- 🔧 **Blueprint it feeds:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) — the production gateway that makes hosted *and* self-hosted models interchangeable behind one interface (the four levers here tell you what each backend costs).
- 🎓 **Capstone:** the cost intuitions here inform [`capstone/llm/gateway.py`](../../../capstone/llm/gateway.py) and its metering (checkpoint `checkpoints/ch39-serving-and-gateway`).
- 📓 **Previous notebook:** [`39-01-llm-gateway-routing-and-fallbacks.ipynb`](39-01-llm-gateway-routing-and-fallbacks.ipynb) — the gateway whose two tiers you just learned to *price*.
- 📖 **Book:** §39.3 (batching, KV-cache, quantization), §39.1–39.2 (hosted vs self-host, VRAM rule, vLLM/TGI/Ollama), §39.4 (speculative decoding), §39.8 (GPU autoscaling & cold starts). Forward: **Ch 40** turns these levers into cost/latency engineering (and the speculative *execution* this chapter warned not to confuse).